In [ ]:
import tiktoken
from torch.utils.data import Dataset,DataLoader
import torch

In [ ]:
### initing random ids
input_token_ids=torch.tensor([2,3,5])

In [ ]:
vocabulary_size=10
n_dim=6 ## dimention of embedding

In [ ]:
embedding=torch.nn.Embedding(vocabulary_size,n_dim)
embedding.weight.size()

In [ ]:
## lookup table for 10 words
embedding.weight

In [ ]:
embedding.weight[input_token_ids]

In [ ]:
## lets make it practical
vocabulary_size=50257 ## max limit for tokenizer for gpt2
n_dim=512
embedding_layer=torch.nn.Embedding(vocabulary_size,n_dim)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self,raw_text,context_size,tokenizer,strides):
        self.input_ids=[]
        self.output_ids=[]
        tokens=tokenizer.encode(raw_text,allowed_special={"<|endoftext|>"})
        for i in range(0,len(tokens)-context_size,strides):
            input_chunk=tokens[i:i+context_size]
            output_chunk=tokens[i+1:i+1+context_size]
            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)    
    def __getitem__(self, index):
        return self.input_ids[index],self.output_ids[index]    
        

In [ ]:
with open("the-verdict.txt","r") as file:
    text=file.read()

In [ ]:
def create_dataloader(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
   ## here max_length=context_size

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = CustomDataset(txt,context_size=max_length,tokenizer=tokenizer, strides=stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
max_length=4 ## context window size is 4
data_loader=create_dataloader(text,batch_size=8,max_length=max_length,stride=max_length)


In [ ]:
data=iter(data_loader)
x,y=next(data)

In [ ]:
x

In [ ]:
token_embedding=embedding_layer(x)

In [ ]:
embedding_layer(x).shape

In [ ]:
### for positional encoding
positional_encoding=torch.nn.Embedding(max_length,n_dim)

In [ ]:
positional_embedding=positional_encoding(torch.arange(max_length)) ## positional encoding for entire batch is same

In [ ]:
## for first batch we have input x
input_embedding=token_embedding+positional_embedding ## adding positional embedding to each token embedding

In [ ]:
input_embedding.size()